In [1]:
import torch
import os
os.chdir('../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6) -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get("inception_feature")
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

In [16]:
!ls samplings/DiT_1.5_7_50000

dpm_0


In [12]:
pt_dir = 'samplings/DiT_1.5_5_50000/dpm_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 5 1.5


100%|██████████| 50001/50001 [00:13<00:00, 3594.89it/s]


22.175060023735284


In [44]:
pt_dir = 'samplings/DiT_1.5_7_50000/dpm_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 7 1.5


100%|██████████| 50001/50001 [00:16<00:00, 3009.24it/s]


7.053009282682638


In [47]:
pt_dir = 'samplings/DiT_1.5_9_50000/dpm_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 9 1.5


100%|██████████| 50001/50001 [00:16<00:00, 3108.94it/s]


4.434725457219031


In [6]:
!ls samplings/dit/euler10k_1.375/

dit_euler10k_1.375_0  dit_euler10k_1.375_1


In [19]:
pt_dir = 'samplings/dit/euler10k_1.375/dit_euler10k_1.375_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


Euler 3 1.375


  7%|▋         | 678/10001 [00:07<01:42, 91.03it/s] 


KeyboardInterrupt: 

In [19]:
pt_dir = 'samplings/dit/euler10k_1.375/dit_euler10k_1.375_1'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


Euler 5 1.375


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:03<00:00, 2933.98it/s]


43.08033099688339


In [20]:
pt_dir = 'samplings/dit/euler10k_1.375/dit_euler10k_1.375_2'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


Euler 7 1.375


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:03<00:00, 2972.48it/s]


20.26963733017766


In [21]:
pt_dir = 'samplings/dit/euler10k_1.375/dit_euler10k_1.375_3'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


Euler 9 1.375


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:03<00:00, 3036.54it/s]


12.115254513023729


In [23]:
!ls samplings/dit/dpm10k_1.375

dit_dpm10k_1.375_0  dit_dpm10k_1.375_1


In [28]:
pt_dir = 'samplings/dit/dpm10k_1.375/dit_dpm10k_1.375_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 5 1.375


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:03<00:00, 2800.26it/s]


29.664809206862742


In [29]:
pt_dir = 'samplings/dit/dpm10k_1.375/dit_dpm10k_1.375_1'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 7 1.375


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:03<00:00, 2930.78it/s]


11.375787472722664


In [35]:
pt_dir = 'samplings/dit/dpm10k_1.375/dit_dpm10k_1.375_2'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 9 1.375


100%|██████████| 10001/10001 [00:03<00:00, 2932.25it/s]


7.555151985946395


### 50k

In [36]:
!ls samplings/dit/dpm50k_1.375/

dit_dpm50k_1.375_0


In [39]:
pt_dir = 'samplings/dit/dpm50k_1.375/dit_dpm50k_1.375_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 5 1.375


100%|██████████| 50001/50001 [00:17<00:00, 2841.89it/s]


27.192424060267


In [44]:
pt_dir = 'samplings/dit/dpm50k_1.375/dit_dpm50k_1.375_1'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 7 1.375


100%|██████████| 50001/50001 [00:17<00:00, 2831.11it/s]


8.728045299711027


In [59]:
pt_dir = 'samplings/dit/dpm50k_1.375/dit_dpm50k_1.375_2'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 9 1.375


100%|██████████| 50001/50001 [00:16<00:00, 3042.17it/s]


4.977645160525071


### DPM 10k

In [86]:
!ls samplings/dit/dpm10k_1.375/

dit_dpm10k_1.375_0  dit_dpm10k_1.375_2	dit_dpm10k_1.375_4  dit_dpm10k_1.375_6
dit_dpm10k_1.375_1  dit_dpm10k_1.375_3	dit_dpm10k_1.375_5


In [87]:
pt_dir = 'samplings/dit/dpm10k_1.375/dit_dpm10k_1.375_6'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 3 1.375


100%|██████████| 10001/10001 [00:02<00:00, 3567.51it/s]


98.45043328203104


In [64]:
pt_dir = 'samplings/dit/dpm10k_1.375/dit_dpm10k_1.375_3'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 3 1.375


100%|██████████| 10001/10001 [00:02<00:00, 3882.66it/s]


98.46516721802436


In [70]:
pt_dir = 'samplings/dit/dpm10k_1.375_raw/dit_dpm10k_1.375_raw_1'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 3 1.375


100%|██████████| 10001/10001 [00:03<00:00, 3214.66it/s]


98.62613887310448


In [73]:
pt_dir = 'samplings/dit/dpm10k_1.375_raw/dit_dpm10k_1.375_raw_3'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 3 1.375


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:02<00:00, 3718.41it/s]


98.62839635054513


In [75]:
pt_dir = 'samplings/dit/dpm10k_1.375_raw/dit_dpm10k_1.375_raw_4'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 3 1.375


100%|██████████| 10001/10001 [00:02<00:00, 3622.56it/s]


99.43063343566371


In [78]:
pt_dir = 'samplings/dit/dpm10k_1.375_raw/dit_dpm10k_1.375_raw_5'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 3 1.375


  0%|          | 0/10001 [00:00<?, ?it/s]

100%|██████████| 10001/10001 [00:02<00:00, 3758.04it/s]


98.91342497049641


### DPM 50k CFG=1.5

In [104]:
!ls samplings/dit/dpm50k_1.5

dpm_0  dpm_1


In [8]:
pt_dir = 'samplings/dit/dpm50k_1.5/dpm_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 5 1.5


 66%|██████▌   | 32879/50001 [00:09<00:04, 3489.77it/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fbe21ae5190>>
Traceback (most recent call last):
  File "/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
100%|██████████| 50001/50001 [00:14<00:00, 3442.42it/s]


22.190812894545104


In [116]:
pt_dir = 'samplings/dit/dpm50k_1.5/dpm_1'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 7 1.5


100%|██████████| 50001/50001 [00:13<00:00, 3666.41it/s]


7.067626923497187


In [125]:
pt_dir = 'samplings/dit/dpm50k_1.5/dpm_2'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)


DPM-Solver 9 1.5


100%|██████████| 50001/50001 [00:13<00:00, 3744.01it/s]


4.437204021123307


In [ ]:
pt_dir = 'samplings/DiT_1.5_100_50000/dpm_0'
config = load_config(pt_dir)
print(config.solver, config.NFE, config.CFG)

data = torch.load('/dataset/dit/stats.pt')
fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
print(fid)

# DPM-Solver 100steps : 2.896